In [0]:
import os
import time
import warnings

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from mlflow import MlflowClient

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [0]:
EXPERIMENT_NAME = "/Shared/Football_MLflow_Experiment"

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Registry URI:", mlflow.get_registry_uri())
print("Experiment:", EXPERIMENT_NAME)

Tracking URI: databricks
Registry URI: databricks-uc
Experiment: /Shared/Football_MLflow_Experiment


In [0]:
DATA_DIRECTORY = "/Volumes/workspace/default/football_data"

TRAIN_PATH = f"{DATA_DIRECTORY}/football_train.csv"
TEST_PATH = f"{DATA_DIRECTORY}/football_test.csv"

CHAMPION_INFO_PATH = (
    f"{DATA_DIRECTORY}/champion_run_info.csv"
)

REGISTRATION_INFO_PATH = (
    f"{DATA_DIRECTORY}/champion_registration_info.csv"
)

REGISTERED_MODEL_NAME = (
    "workspace.default.football_match_result_model"
)

CHAMPION_ALIAS = "champion"

TARGET_COLUMN = "match_result"

print("Registered model name:", REGISTERED_MODEL_NAME)
print("Champion alias:", CHAMPION_ALIAS)

Registered model name: workspace.default.football_match_result_model
Champion alias: champion


In [0]:
assert os.path.exists(CHAMPION_INFO_PATH), (
    "champion_run_info.csv was not found. "
    "Run Notebook 3 before continuing."
)

champion_info_df = pd.read_csv(
    CHAMPION_INFO_PATH
)

display(champion_info_df)

model_role,algorithm,run_id,model_uri,primary_metric,primary_metric_value,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,training_rows,testing_rows,training_time_seconds,max_depth,min_samples_split,min_samples_leaf,class_weight,removed_columns_path,registration_status,alias_status
champion,DecisionTreeClassifier,3fa9c04ecf3d4f1597aa3bd68f650d2f,models:/m-a7c158f4672f45faad9513d569e68b0c,test_f1_macro,0.3407115240453506,0.35575,0.5227480171428843,0.4249912752465563,0.3407115240453506,0.3219326636027983,15999,4000,0.334583044052124,3,20,10,balanced,/Volumes/workspace/default/football_data/champion_removed_columns.json,pending,pending


In [0]:
required_columns = [
    "algorithm",
    "run_id",
    "model_uri",
    "primary_metric",
    "primary_metric_value"
]

missing_columns = [
    column
    for column in required_columns
    if column not in champion_info_df.columns
]

assert not missing_columns, (
    "The following required columns are missing: "
    f"{missing_columns}"
)

assert len(champion_info_df) == 1, (
    "champion_run_info.csv should contain exactly one row."
)

print("Champion information validated successfully.")

Champion information validated successfully.


In [0]:
champion_algorithm = str(
    champion_info_df.iloc[0]["algorithm"]
)

assert champion_algorithm == "DecisionTreeClassifier", (
    "Notebook 4 expected a DecisionTreeClassifier, "
    f"but found: {champion_algorithm}"
)

print("Champion algorithm:", champion_algorithm)

Champion algorithm: DecisionTreeClassifier


In [0]:
champion_record = champion_info_df.iloc[0]

champion_run_id = str(
    champion_record["run_id"]
)

champion_model_uri = str(
    champion_record["model_uri"]
)

champion_primary_metric = str(
    champion_record["primary_metric"]
)

champion_primary_metric_value = float(
    champion_record["primary_metric_value"]
)

print("Algorithm:", champion_algorithm)
print("Run ID:", champion_run_id)
print("Logged model URI:", champion_model_uri)
print("Primary metric:", champion_primary_metric)

print(
    "Primary metric value:",
    round(champion_primary_metric_value, 4)
)

Algorithm: DecisionTreeClassifier
Run ID: 3fa9c04ecf3d4f1597aa3bd68f650d2f
Logged model URI: models:/m-a7c158f4672f45faad9513d569e68b0c
Primary metric: test_f1_macro
Primary metric value: 0.3407


In [0]:
logged_champion_model = mlflow.sklearn.load_model(
    champion_model_uri
)

print("Logged Champion model loaded successfully.")
print("Loaded model type:", type(logged_champion_model))

Logged Champion model loaded successfully.
Loaded model type: <class 'sklearn.pipeline.Pipeline'>


In [0]:
test_df = pd.read_csv(TEST_PATH)

assert TARGET_COLUMN in test_df.columns, (
    f"{TARGET_COLUMN} is missing from the test dataset."
)

LEAKAGE_COLUMNS = [
    "home_score",
    "away_score"
]

X_test = test_df.drop(
    columns=[
        TARGET_COLUMN,
        *[
            column
            for column in LEAKAGE_COLUMNS
            if column in test_df.columns
        ]
    ],
    errors="ignore"
).copy()

print("Test feature shape:", X_test.shape)

Test feature shape: (4000, 57)


In [0]:
pre_registration_sample = X_test.head(10).copy()

pre_registration_predictions = (
    logged_champion_model.predict(
        pre_registration_sample
    )
)

print("Pre-registration predictions:")
print(pre_registration_predictions)

assert len(pre_registration_predictions) == 10

print("Pre-registration inference succeeded.")

Pre-registration predictions:
['Draw' 'Draw' 'Draw' 'Draw' 'Draw' 'Draw' 'Away Win' 'Draw' 'Home Win'
 'Draw']
Pre-registration inference succeeded.


In [0]:
client = MlflowClient()

print("MLflow client created successfully.")

MLflow client created successfully.


In [0]:
try:
    existing_registered_model = (
        client.get_registered_model(
            REGISTERED_MODEL_NAME
        )
    )

    print("Registered model already exists.")
    print(
        "Registered model:",
        existing_registered_model.name
    )

except Exception as error:
    existing_registered_model = None

    print(
        "Registered model does not exist yet "
        "or could not be retrieved."
    )

    print("Details:", str(error))

Registered model already exists.
Registered model: workspace.default.football_match_result_model


In [0]:
previous_champion_version = None
previous_champion_run_id = None

try:
    previous_alias_model = (
        client.get_model_version_by_alias(
            name=REGISTERED_MODEL_NAME,
            alias=CHAMPION_ALIAS
        )
    )

    previous_champion_version = str(
        previous_alias_model.version
    )

    previous_champion_run_id = str(
        previous_alias_model.run_id
    )

    print("Existing champion alias found.")
    print(
        "Previous Champion version:",
        previous_champion_version
    )
    print(
        "Previous Champion run ID:",
        previous_champion_run_id
    )

except Exception:
    print(
        "No existing champion alias was found."
    )

Existing champion alias found.
Previous Champion version: 3
Previous Champion run ID: 274498f94f3145bdbfcc664a5dd9a952


In [0]:
print("Registering the Decision Tree Champion...")

registration_result = mlflow.register_model(
    model_uri=champion_model_uri,
    name=REGISTERED_MODEL_NAME
)

registered_model_version = str(
    registration_result.version
)

print("Model registration request completed.")
print("Registered model:", REGISTERED_MODEL_NAME)
print("New model version:", registered_model_version)
print("Registration status:", registration_result.status)

Registering the Decision Tree Champion...


Registered model 'workspace.default.football_match_result_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '4' of model 'workspace.default.football_match_result_model': https://dbc-f41d3394-c5eb.cloud.databricks.com/explore/data/models/workspace/default/football_match_result_model/version/4?o=7474658572124013


Model registration request completed.
Registered model: workspace.default.football_match_result_model
New model version: 4
Registration status: READY


In [0]:
MAX_WAIT_SECONDS = 180
POLL_INTERVAL_SECONDS = 5

waited_seconds = 0

while waited_seconds < MAX_WAIT_SECONDS:

    model_version_details = (
        client.get_model_version(
            name=REGISTERED_MODEL_NAME,
            version=registered_model_version
        )
    )

    current_status = str(
        model_version_details.status
    )

    print(
        f"Model version status: {current_status}"
    )

    if current_status == "READY":
        print("Model version is ready.")
        break

    if current_status == "FAILED_REGISTRATION":
        raise RuntimeError(
            "Model registration failed. "
            f"Status message: "
            f"{model_version_details.status_message}"
        )

    time.sleep(POLL_INTERVAL_SECONDS)

    waited_seconds += POLL_INTERVAL_SECONDS

else:
    raise TimeoutError(
        "The model version did not become READY "
        f"within {MAX_WAIT_SECONDS} seconds."
    )

Model version status: READY
Model version is ready.


In [0]:
registered_model_description = (
    "Football match-result classification model used for the "
    "Champion-Challenger workflow. The registered model stores "
    "the initial Decision Tree Champion and later Challenger "
    "versions evaluated using Macro F1."
)

client.update_registered_model(
    name=REGISTERED_MODEL_NAME,
    description=registered_model_description
)

print("Registered-model description updated.")

Registered-model description updated.


In [0]:
if (
    previous_champion_version is not None
    and previous_champion_version
    != registered_model_version
):

    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=previous_champion_version,
        key="previous_alias_status",
        value="replaced_during_version_2_redesign"
    )

    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=previous_champion_version,
        key="replaced_by_version",
        value=registered_model_version
    )

    print(
        "Previous Champion version tagged successfully."
    )

else:
    print(
        "No different previous Champion version "
        "required tagging."
    )

Previous Champion version tagged successfully.


In [0]:
decision_tree_tags = {
    "model_role": "champion",
    "algorithm": "DecisionTreeClassifier",
    "workflow": "champion_challenger",
    "workflow_version": "version_2",
    "model_stage": "initial_baseline",
    "primary_metric": champion_primary_metric,
    "primary_metric_value": str(
        champion_primary_metric_value
    ),
    "registration_notebook": (
        "04_Champion_Registration"
    ),
    "promotion_status": "current_champion"
}

for tag_key, tag_value in decision_tree_tags.items():

    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=registered_model_version,
        key=tag_key,
        value=str(tag_value)
    )

print("Decision Tree model-version tags added.")

Decision Tree model-version tags added.


In [0]:
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias=CHAMPION_ALIAS,
    version=registered_model_version
)

print(
    f"Alias '{CHAMPION_ALIAS}' assigned to "
    f"version {registered_model_version}."
)

Alias 'champion' assigned to version 4.


In [0]:
alias_model_version = (
    client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias=CHAMPION_ALIAS
    )
)

resolved_alias_version = str(
    alias_model_version.version
)

resolved_alias_run_id = str(
    alias_model_version.run_id
)

print("Alias verification results:")
print("Alias:", CHAMPION_ALIAS)
print("Resolved version:", resolved_alias_version)
print("Resolved run ID:", resolved_alias_run_id)

Alias verification results:
Alias: champion
Resolved version: 4
Resolved run ID: 3fa9c04ecf3d4f1597aa3bd68f650d2f


In [0]:
assert (
    resolved_alias_version
    == registered_model_version
), (
    "The champion alias does not point to "
    "the newly registered Decision Tree version."
)

assert resolved_alias_run_id == champion_run_id, (
    "The run ID resolved through the alias does "
    "not match the Notebook 3 Champion run."
)

print(
    "The champion alias correctly points to "
    "the Decision Tree version."
)

The champion alias correctly points to the Decision Tree version.


In [0]:
CHAMPION_ALIAS_MODEL_URI = (
    f"models:/{REGISTERED_MODEL_NAME}"
    f"@{CHAMPION_ALIAS}"
)

print(
    "Champion alias model URI:",
    CHAMPION_ALIAS_MODEL_URI
)

Champion alias model URI: models:/workspace.default.football_match_result_model@champion


In [0]:
registered_champion_model = (
    mlflow.sklearn.load_model(
        CHAMPION_ALIAS_MODEL_URI
    )
)

print(
    "Registered Champion loaded successfully "
    "through the alias."
)

print(
    "Loaded object type:",
    type(registered_champion_model)
) 

Registered Champion loaded successfully through the alias.
Loaded object type: <class 'sklearn.pipeline.Pipeline'>


In [0]:
alias_predictions = (
    registered_champion_model.predict(
        pre_registration_sample
    )
)

print("Alias-based predictions:")
print(alias_predictions)

Alias-based predictions:
['Draw' 'Draw' 'Draw' 'Draw' 'Draw' 'Draw' 'Away Win' 'Draw' 'Home Win'
 'Draw']


In [0]:
predictions_match = np.array_equal(
    pre_registration_predictions,
    alias_predictions
)

print(
    "Predictions match:",
    predictions_match
)

assert predictions_match, (
    "Predictions from the registered model do "
    "not match the original logged model."
)

print(
    "Original and alias-loaded models produced "
    "identical predictions."
)

Predictions match: True
Original and alias-loaded models produced identical predictions.


In [0]:
final_model_version_details = (
    client.get_model_version(
        name=REGISTERED_MODEL_NAME,
        version=registered_model_version
    )
)

print("Registered model:", final_model_version_details.name)
print("Version:", final_model_version_details.version)
print("Status:", final_model_version_details.status)
print("Run ID:", final_model_version_details.run_id)
print("Aliases:", final_model_version_details.aliases)
print("Description:", final_model_version_details.description)
print("Tags:", final_model_version_details.tags)

Registered model: workspace.default.football_match_result_model
Version: 4
Status: READY
Run ID: 3fa9c04ecf3d4f1597aa3bd68f650d2f
Aliases: ['champion']
Description: 
Tags: {'algorithm': 'DecisionTreeClassifier', 'model_role': 'champion', 'model_stage': 'initial_baseline', 'primary_metric': 'test_f1_macro', 'primary_metric_value': '0.3407115240453506', 'promotion_status': 'current_champion', 'registration_notebook': '04_Champion_Registration', 'workflow': 'champion_challenger', 'workflow_version': 'version_2'}


In [0]:
registration_information = pd.DataFrame(
    [
        {
            "registered_model_name": (
                REGISTERED_MODEL_NAME
            ),
            "registered_model_version": (
                registered_model_version
            ),
            "alias": CHAMPION_ALIAS,
            "alias_model_uri": (
                CHAMPION_ALIAS_MODEL_URI
            ),
            "algorithm": champion_algorithm,
            "run_id": champion_run_id,
            "source_model_uri": (
                champion_model_uri
            ),
            "primary_metric": (
                champion_primary_metric
            ),
            "primary_metric_value": (
                champion_primary_metric_value
            ),
            "registration_status": "READY",
            "alias_status": "assigned",
            "model_role": "champion",
            "previous_champion_version": (
                previous_champion_version
            ),
            "previous_champion_run_id": (
                previous_champion_run_id
            )
        }
    ]
)

registration_information.to_csv(
    REGISTRATION_INFO_PATH,
    index=False
)

display(registration_information)

print(
    "Registration information saved to:"
)

print(REGISTRATION_INFO_PATH)

registered_model_name,registered_model_version,alias,alias_model_uri,algorithm,run_id,source_model_uri,primary_metric,primary_metric_value,registration_status,alias_status,model_role,previous_champion_version,previous_champion_run_id
workspace.default.football_match_result_model,4,champion,models:/workspace.default.football_match_result_model@champion,DecisionTreeClassifier,3fa9c04ecf3d4f1597aa3bd68f650d2f,models:/m-a7c158f4672f45faad9513d569e68b0c,test_f1_macro,0.3407115240453506,READY,assigned,champion,3,274498f94f3145bdbfcc664a5dd9a952


Registration information saved to:
/Volumes/workspace/default/football_data/champion_registration_info.csv


In [0]:
champion_info_df.loc[
    0,
    "registered_model_name"
] = REGISTERED_MODEL_NAME

champion_info_df.loc[
    0,
    "registered_model_version"
] = registered_model_version

champion_info_df.loc[
    0,
    "registered_model_alias"
] = CHAMPION_ALIAS

champion_info_df.loc[
    0,
    "alias_model_uri"
] = CHAMPION_ALIAS_MODEL_URI

champion_info_df.loc[
    0,
    "registration_status"
] = "READY"

champion_info_df.loc[
    0,
    "alias_status"
] = "assigned"

champion_info_df.to_csv(
    CHAMPION_INFO_PATH,
    index=False
)

display(champion_info_df)

print(
    "champion_run_info.csv updated successfully."
)

model_role,algorithm,run_id,model_uri,primary_metric,primary_metric_value,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted,training_rows,testing_rows,training_time_seconds,max_depth,min_samples_split,min_samples_leaf,class_weight,removed_columns_path,registration_status,alias_status,registered_model_name,registered_model_version,registered_model_alias,alias_model_uri
champion,DecisionTreeClassifier,3fa9c04ecf3d4f1597aa3bd68f650d2f,models:/m-a7c158f4672f45faad9513d569e68b0c,test_f1_macro,0.3407115240453506,0.35575,0.5227480171428843,0.4249912752465563,0.3407115240453506,0.3219326636027983,15999,4000,0.334583044052124,3,20,10,balanced,/Volumes/workspace/default/football_data/champion_removed_columns.json,READY,assigned,workspace.default.football_match_result_model,4,champion,models:/workspace.default.football_match_result_model@champion


champion_run_info.csv updated successfully.


In [0]:
assert os.path.exists(
    CHAMPION_INFO_PATH
), "champion_run_info.csv is missing."

assert os.path.exists(
    REGISTRATION_INFO_PATH
), "champion_registration_info.csv is missing."

saved_champion_info = pd.read_csv(
    CHAMPION_INFO_PATH
)

saved_registration_info = pd.read_csv(
    REGISTRATION_INFO_PATH
)

assert len(saved_champion_info) == 1
assert len(saved_registration_info) == 1

assert str(
    saved_champion_info.iloc[0][
        "registered_model_version"
    ]
) == registered_model_version

assert (
    saved_champion_info.iloc[0][
        "registered_model_alias"
    ]
    == CHAMPION_ALIAS
)

assert (
    saved_registration_info.iloc[0][
        "algorithm"
    ]
    == "DecisionTreeClassifier"
)

print("Saved files validated successfully.")

Saved files validated successfully.


In [0]:
final_alias_check = (
    client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias=CHAMPION_ALIAS
    )
)

assert str(
    final_alias_check.version
) == registered_model_version

assert str(
    final_alias_check.run_id
) == champion_run_id

assert CHAMPION_ALIAS in (
    final_alias_check.aliases
)

print(
    "Final registry verification completed."
)

Final registry verification completed.


In [0]:
print("=" * 70)
print("NOTEBOOK 4 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME
)

print(
    "Algorithm:",
    champion_algorithm
)

print(
    "Model version:",
    registered_model_version
)

print(
    "Alias:",
    CHAMPION_ALIAS
)

print(
    "Alias model URI:",
    CHAMPION_ALIAS_MODEL_URI
)

print(
    "Run ID:",
    champion_run_id
)

print(
    "Champion Macro F1:",
    round(
        champion_primary_metric_value,
        4
    )
)

print(
    "Registration status:",
    "READY"
)

print(
    "Alias verification:",
    "PASSED"
)

print(
    "Saved registration file:",
    REGISTRATION_INFO_PATH
)

print(
    "Updated Champion file:",
    CHAMPION_INFO_PATH
)

print(
    "Result: The Decision Tree is now the "
    "registered baseline Champion."
)

NOTEBOOK 4 COMPLETED SUCCESSFULLY
Registered model: workspace.default.football_match_result_model
Algorithm: DecisionTreeClassifier
Model version: 4
Alias: champion
Alias model URI: models:/workspace.default.football_match_result_model@champion
Run ID: 3fa9c04ecf3d4f1597aa3bd68f650d2f
Champion Macro F1: 0.3407
Registration status: READY
Alias verification: PASSED
Saved registration file: /Volumes/workspace/default/football_data/champion_registration_info.csv
Updated Champion file: /Volumes/workspace/default/football_data/champion_run_info.csv
Result: The Decision Tree is now the registered baseline Champion.
